# Day1: Count Matrix QC

このノートブックでは、RNA-seqの最初の入口である `count matrix` を理解します。目的は、行と列が何を表し、なぜ最初にサンプル全体の量を確認するのかをつかむことです。

このトピックが役立つ場面:
炎症刺激をかけた細胞の RNA-seq 結果を受け取ったときに、まず「この表は何を数えていて、刺激群と対照群を比べる出発点として使えるのか」を判断したい場面です。

このトピックで解く課題:
`IL6` が刺激群で高そうに見えるが、その前にこのデータ自体を信じて先へ進めてよいかを見極めます。

## まず、なぜ比較するのか

RNA-seqで本当に知りたいのは、たいてい「条件が変わると、細胞の中でどの遺伝子の使われ方が変わるのか」です。

この教材では、次の問いを考えます。

`炎症刺激を加えた細胞では、加えていない細胞と比べて、どの遺伝子のRNAが増えたり減ったりしたのか`

ここで大事なのは、1つのサンプルだけを見るのではなく、条件の違うサンプル同士を比較することです。比較が必要な理由は、細胞の状態変化を見つけたいからです。

## RNA-seq は何を数えているのか

細胞の中では、DNAの情報が必要に応じてRNAとして読み出されます。RNA-seqは、そのRNAを短い断片として読み取り、どの遺伝子由来の断片が何個あったかを数える実験です。

つまり、`count matrix` の数値は「その遺伝子のRNA断片が何個読まれたか」を表しています。数が大きいほど、その条件でその遺伝子がよく使われている可能性があります。

## なぜ count matrix から始めるのか

データサイエンスの視点で見ると、`count matrix` は特徴量テーブルに近い形をしています。

- 行: 遺伝子
- 列: サンプル
- 値: 観測された count

ただし、普通の表形式データと違って、各サンプルの合計量がそろっていないことが多いです。そのため、いきなり値の大小だけを見て結論を出すことはできません。まずは、この比較が公平にできそうかを確かめます。

## count matrix の読み方

- 行: 遺伝子
- 列: サンプル
- 値: RNA断片の数

例えば `IL6` の値が control で低く、treated で高ければ、treated 条件で `IL6` のRNAが増えていそうだと考えます。

In [ ]:
# pandas: 表形式データを扱うためのライブラリ
# matplotlib: 結果を図として確認するためのライブラリ
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 例として使う count matrix

In [ ]:
# 行は遺伝子、列はサンプル、値はRNA断片の数です。
# control は刺激なし、treated は炎症刺激ありのサンプルだと考えます。
counts = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "control_1": [120, 300, 5000, 4200, 30],
    "control_2": [100, 280, 5100, 4000, 25],
    "treated_1": [300, 600, 5300, 4100, 200],
    "treated_2": [280, 650, 5200, 4300, 220],
}).set_index("gene")

counts

## まず確認すること: library size

RNA-seqでは、サンプルごとに読まれた総量が違うことがあります。この総量を `library size` と呼びます。生の count をそのまま比較する前に、まずここを確認します。

In [ ]:
# library size は、各サンプルで読まれたRNA断片数の合計です。
# axis=0 は「列ごとに合計する」という意味です。
library_size = counts.sum(axis=0)
library_size

In [ ]:
# サンプルごとの総read数を棒グラフで確認します。
# ここに大きな差があると、生のcountをそのまま比較しにくくなります。
plt.figure(figsize=(6, 4))
library_size.plot(kind="bar", color=["#4C78A8", "#4C78A8", "#F58518", "#F58518"])
plt.ylabel("Total counts")
plt.title("Library size per sample")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

この例では treated サンプルの総カウント数がやや大きく、control と完全にはそろっていません。ここを無視すると、「本当に遺伝子発現が高いのか」「単にたくさん読まれただけなのか」が区別しにくくなります。

## どの遺伝子が増えていそうかを見る

正式な差次的発現解析ではありませんが、平均値を使ってざっくり変化の方向を見ることはできます。

In [ ]:
# 条件ごとに平均を取り、遺伝子ごとの大まかな変化方向を見ます。
# axis=1 は「行ごとに平均する」という意味です。
treated_mean = counts[["treated_1", "treated_2"]].mean(axis=1)
control_mean = counts[["control_1", "control_2"]].mean(axis=1)
# rough_fold_change は「刺激あり / 刺激なし」のおおまかな比です。
# +1 は、0で割ることを避けるための簡単な処理です。
summary = pd.DataFrame({
    "control_mean": control_mean,
    "treated_mean": treated_mean,
    "rough_fold_change": (treated_mean + 1) / (control_mean + 1)
}).sort_values("rough_fold_change", ascending=False)

summary

`IL6` は treated で大きく増えており、この段階でも注目候補だと分かります。ただし、最終的には正規化と統計解析を通して判断します。

## ここまでの要点

- count matrix は「遺伝子ごとのRNA断片数の表」
- 行は遺伝子、列はサンプル、値は count
- まず library size を見て、サンプル全体の量の差を確認する
- 次のステップは正規化